In [0]:
# Imports and logger setup
import sys
sys.path.append("/Workspace/claim-denial-prevention")

from logger import (
    get_logger, log_start, log_end,
    log_step, log_success, log_warning, log_error
)
from error_codes import ErrorCodes

logger = get_logger("silver_cleaning")
log_start(logger, "silver_cleaning")

### Configuration

In [0]:
try:
    BASE_PATH   = "/Volumes/workspace/default/myvol"

    BRONZE_PATHS = {
        "claims":    f"{BASE_PATH}/bronze/claims/",
        "providers": f"{BASE_PATH}/bronze/providers/",
        "diagnosis": f"{BASE_PATH}/bronze/diagnosis/",
        "cost":      f"{BASE_PATH}/bronze/cost/",
    }

    SILVER_PATH = f"{BASE_PATH}/silver/claims_cleaned/"

    # Required columns in final Silver table
    REQUIRED_SILVER_COLS = [
        "claim_id", "patient_id", "provider_id", "claim_date", "claim_age_days",
        "diagnosis_code", "procedure_code",
        "billed_amount", "average_cost", "expected_cost",
        "specialty", "location",
        "diagnosis_category", "diagnosis_severity",
        "is_diagnosis_missing", "is_procedure_missing",
        "denial_flag",
        "_silver_timestamp", "_silver_layer"
    ]

    log_success(logger, "Configuration loaded")
    logger.info(f"Silver output path: {SILVER_PATH}")

except Exception as e:
    log_error(logger, ErrorCodes.INF_001, ErrorCodes.get_message(ErrorCodes.INF_001), e)
    raise

### Load all 4 bronze tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

log_step(logger, 1, "Loading Bronze Delta tables")

dataframes = {}

for name, path in BRONZE_PATHS.items():
    try:
        logger.info(f"Loading '{name}' from {path}")
        df = spark.read.format("delta").load(path)
        dataframes[name] = df
        log_success(logger, f"[{name}] loaded — {df.count()} rows × {len(df.columns)} columns")
    except Exception as e:
        log_error(
            logger, ErrorCodes.SLV_001,
            f"Bronze table '{name}' not found at {path}. "
            f"{ErrorCodes.get_message(ErrorCodes.SLV_001)}", e
        )
        raise RuntimeError(
            ErrorCodes.format_error(ErrorCodes.SLV_001, f"Missing table: '{name}'")
        )

df_claims    = dataframes["claims"]
df_providers = dataframes["providers"]
df_diagnosis = dataframes["diagnosis"]
df_cost      = dataframes["cost"]

logger.info(f"Starting Silver pipeline with {df_claims.count()} raw claims")

### Deduplication

In [0]:
# Cell 4 — Step 1: Skip deduplication (EDA confirmed 0 duplicates)
# Documented here for pipeline traceability
log_step(logger, 2, "Deduplication Check")

try:
    total  = df_claims.count()
    unique = df_claims.select("claim_id").distinct().count()
    dupes  = total - unique

    logger.info(f"Duplicate check: {total} total, {unique} unique, {dupes} duplicates")

    if dupes == 0:
        log_success(logger, "No duplicates found — skipping dedup step (confirmed by EDA)")
        df_deduped = df_claims
    else:
        # Fallback: run dedup if somehow duplicates appear (e.g. re-ingestion)
        log_warning(
            logger, ErrorCodes.SLV_002,
            f"Unexpected {dupes} duplicates found — running deduplication"
        )
        window_spec = Window.partitionBy("claim_id") \
                            .orderBy(F.col("_ingestion_timestamp").desc())
        df_deduped = df_claims \
            .withColumn("_rank", F.rank().over(window_spec)) \
            .filter(F.col("_rank") == 1) \
            .drop("_rank")

        log_success(logger, f"Deduplication complete — {df_deduped.count()} rows remaining")

    logger.info(f"After dedup step: {df_deduped.count()} claims")

except Exception as e:
    log_error(logger, ErrorCodes.SLV_002, ErrorCodes.get_message(ErrorCodes.SLV_002), e)
    raise

### Handle nulls

In [0]:
log_step(logger, 3, "Null Handling")

try:
    logger.info("Imputing null diagnosis_code → 'UNKNOWN' (307 expected)")
    logger.info("Imputing null procedure_code → 'UNKNOWN' (241 expected)")
    logger.info("Imputing null billed_amount  → 0.0 (343 expected)")

    df_nulls_handled = df_deduped \
        .withColumn(
            "diagnosis_code",
            F.when(F.col("diagnosis_code").isNull(), F.lit("UNKNOWN"))
             .otherwise(F.col("diagnosis_code"))
        ) \
        .withColumn(
            "procedure_code",
            F.when(F.col("procedure_code").isNull(), F.lit("UNKNOWN"))
             .otherwise(F.col("procedure_code"))
        ) \
        .withColumn(
            "billed_amount",
            F.when(
                F.col("billed_amount").isNull() | (F.col("billed_amount") <= 0),
                F.lit(0.0)
            ).otherwise(F.col("billed_amount"))
        )

    # Validate imputation worked
    remaining_null_diag = df_nulls_handled.filter(F.col("diagnosis_code").isNull()).count()
    remaining_null_proc = df_nulls_handled.filter(F.col("procedure_code").isNull()).count()
    remaining_null_amt  = df_nulls_handled.filter(F.col("billed_amount").isNull()).count()

    if remaining_null_diag > 0:
        log_warning(logger, ErrorCodes.SLV_003,
                    f"{remaining_null_diag} diagnosis_code nulls remain after imputation")
    else:
        log_success(logger, "diagnosis_code: all nulls imputed to UNKNOWN")

    if remaining_null_proc > 0:
        log_warning(logger, ErrorCodes.SLV_003,
                    f"{remaining_null_proc} procedure_code nulls remain after imputation")
    else:
        log_success(logger, "procedure_code: all nulls imputed to UNKNOWN")

    if remaining_null_amt > 0:
        log_warning(logger, ErrorCodes.SLV_003,
                    f"{remaining_null_amt} billed_amount nulls remain after imputation")
    else:
        log_success(logger, "billed_amount: all nulls imputed to 0.0")

    logger.info(f"After null handling: {df_nulls_handled.count()} claims")

except Exception as e:
    log_error(logger, ErrorCodes.SLV_003, ErrorCodes.get_message(ErrorCodes.SLV_003), e)
    raise

## Generate Synthetic Labels

In [0]:
# Cell 6 — Synthetic Label Generation
# Target denial rate : ~30% (300 denied / 700 approved)
# Method             : Weighted points system — all features contribute
# Algorithms         : Designed for XGBoost + Logistic Regression
#
# POINTS TABLE:
#   Missing diagnosis_code          → 3 pts  (strong denial signal)
#   Missing procedure_code          → 3 pts  (strong denial signal)
#   Missing billed_amount (=0.0)    → 2 pts  (moderate denial signal)
#   Overbilling (> avg_cost * 2)    → applied post cost-join in Cell 10b
#   Provider specialty risk         → 1 pt   (weak signal, applied here)
#
# THRESHOLD: points >= 3 → denied
#   This naturally hits ~30% given EDA null rates:
#   307 null diagnosis (30.7%) + 241 null procedure (24.1%)
#   with overlap handled by the points sum

log_step(logger, 4, "Synthetic Label Generation — Weighted Points System")

try:
    from pyspark.sql.types import IntegerType, DoubleType, StringType

    logger.info("Target denial rate: ~30% (300/1000)")
    logger.info("Method: weighted points system — threshold >= 3 points = denied")
    logger.info("Designed for XGBoost + Logistic Regression classification")

    # High-risk specialties identified from domain knowledge
    # These providers historically have higher denial rates
    HIGH_RISK_SPECIALTIES = [
        "Cardiology", "Oncology", "Neurology",
        "Orthopedics", "Pain Management"
    ]

    df_with_points = df_nulls_handled \
        .withColumn(
            # ── DENIAL POINTS ────────────────────────────────────────────
            # Sum of all risk signals — higher = more likely denied
            "denial_points",
            (
                # Missing diagnosis → 3 pts
                F.when(
                    F.col("diagnosis_code") == "UNKNOWN",
                    F.lit(3)
                ).otherwise(F.lit(0))
                +
                # Missing procedure → 3 pts
                F.when(
                    F.col("procedure_code") == "UNKNOWN",
                    F.lit(3)
                ).otherwise(F.lit(0))
                +
                # Missing billed amount → 2 pts
                F.when(
                    F.col("billed_amount") == 0.0,
                    F.lit(2)
                ).otherwise(F.lit(0))
            ).cast(IntegerType())
        ) \
        .withColumn(
            # ── DENIAL REASON CODE ───────────────────────────────────────
            # Primary reason — RAG uses this to retrieve policy chunks
            # Highest-weight rule wins if multiple triggered
            "denial_reason_code",
            F.when(
                F.col("diagnosis_code") == "UNKNOWN",
                F.lit("R01_MISSING_DIAGNOSIS")
            ).when(
                F.col("procedure_code") == "UNKNOWN",
                F.lit("R02_MISSING_PROCEDURE")
            ).when(
                F.col("billed_amount") == 0.0,
                F.lit("R03_MISSING_BILLED_AMOUNT")
            ).otherwise(
                F.lit("NONE")   # will be updated post cost-join
            )
        )

    # ── Preview points distribution before thresholding ──────────────────────
    logger.info("Points distribution before thresholding:")
    df_with_points.groupBy("denial_points") \
        .count() \
        .orderBy("denial_points") \
        .show()

    # ── Apply threshold → binary label ────────────────────────────────────────
    # Threshold = 3: claims with 3+ points are denied
    # This hits ~30% denial rate given our EDA null rates
    DENIAL_THRESHOLD = 3

    df_with_denial = df_with_points \
        .withColumn(
            # ── DENIAL FLAG (binary target for ML) ───────────────────────
            "denial_flag",
            F.when(
                F.col("denial_points") >= DENIAL_THRESHOLD,
                F.lit(1)
            ).otherwise(F.lit(0)).cast(IntegerType())
        ) \
        .withColumn(
            # ── DENIAL PROBABILITY (soft score anchor for ML calibration) ─
            # Scaled from points: 0 pts → 0.05, 8 pts → 0.95
            # ML model will refine these in Week 3
            "denial_probability",
            F.when(
                F.col("denial_points") == 0, F.lit(0.05)
            ).when(
                F.col("denial_points") == 1, F.lit(0.20)
            ).when(
                F.col("denial_points") == 2, F.lit(0.40)
            ).when(
                F.col("denial_points") == 3, F.lit(0.65)
            ).when(
                F.col("denial_points") == 4, F.lit(0.78)
            ).when(
                F.col("denial_points") == 5, F.lit(0.88)
            ).otherwise(
                F.lit(0.95)  # 6+ points → near certain denial
            ).cast(DoubleType())
        ) \
        .withColumn(
            # ── RISK TIER (matches FastAPI response schema) ───────────────
            "risk_tier",
            F.when(
                F.col("denial_probability") >= 0.70, F.lit("HIGH")
            ).when(
                F.col("denial_probability") >= 0.40, F.lit("MEDIUM")
            ).otherwise(F.lit("LOW"))
        )

    # ── Validate denial rate ──────────────────────────────────────────────────
    total          = df_with_denial.count()
    denied         = df_with_denial.filter(F.col("denial_flag") == 1).count()
    approved       = total - denied
    denial_rate    = round(denied / total * 100, 2)

    logger.info(f"Denial threshold used: {DENIAL_THRESHOLD} points")
    logger.info(f"Denial rate achieved : {denial_rate}% (target: ~30%)")

    print("\n" + "=" * 60)
    print("SYNTHETIC LABEL DISTRIBUTION — BEFORE COST JOIN")
    print("=" * 60)
    print(f"  Denial threshold  : >= {DENIAL_THRESHOLD} points")
    print(f"  Total claims      : {total}")
    print(f"  Denied  (flag=1)  : {denied}  ({denial_rate}%)")
    print(f"  Approved (flag=0) : {approved} ({round(100 - denial_rate, 2)}%)")
    print("=" * 60)

    print("\n  Denial reason breakdown (pre cost-join):")
    df_with_denial.groupBy("denial_reason_code") \
        .count() \
        .orderBy(F.col("count").desc()) \
        .show(truncate=False)

    print("\n  Risk tier distribution:")
    df_with_denial.groupBy("risk_tier") \
        .count() \
        .orderBy("risk_tier") \
        .show()

    # ── Warn if off target ────────────────────────────────────────────────────
    if denial_rate < 20:
        log_warning(
            logger, ErrorCodes.EDA_005,
            f"Denial rate ({denial_rate}%) is below 20% — "
            f"consider lowering DENIAL_THRESHOLD to 2"
        )
    elif denial_rate > 45:
        log_warning(
            logger, ErrorCodes.EDA_005,
            f"Denial rate ({denial_rate}%) is above 45% — "
            f"consider raising DENIAL_THRESHOLD to 4"
        )
    else:
        log_success(
            logger,
            f"Label distribution looks good — "
            f"denial rate: {denial_rate}% (target: ~30%)"
        )

except Exception as e:
    log_error(
        logger, ErrorCodes.SLV_003,
        "Synthetic label generation failed", e
    )
    raise

### Standardise codes

In [0]:
# Cell 7 — Step 4: Standardize codes
# EDA confirmed 0 format issues — but we still apply as defensive measure

log_step(logger, 5, "Code Standardization (Defensive)")

try:
    df_standardized = df_with_denial \
        .withColumn("diagnosis_code", F.upper(F.trim(F.col("diagnosis_code")))) \
        .withColumn("procedure_code", F.upper(F.trim(F.col("procedure_code"))))

    # Parse claim_date to proper DateType
    try:
        df_standardized = df_standardized \
            .withColumn("claim_date", F.to_date(F.col("claim_date"), "yyyy-MM-dd"))

        null_dates = df_standardized.filter(F.col("claim_date").isNull()).count()
        if null_dates > 0:
            log_warning(
                logger, ErrorCodes.SLV_011,
                f"{null_dates} claim_date values failed to parse — "
                f"check if date format matches yyyy-MM-dd"
            )
        else:
            log_success(logger, "claim_date parsed to DateType successfully")

    except Exception as date_err:
        log_error(logger, ErrorCodes.SLV_011,
                  ErrorCodes.get_message(ErrorCodes.SLV_011), date_err)
        raise

    # Add missing code flag columns (these become ML features in Week 3)
    df_standardized = df_standardized \
        .withColumn(
            "is_diagnosis_missing",
            (F.col("diagnosis_code") == "UNKNOWN").cast(IntegerType())
        ) \
        .withColumn(
            "is_procedure_missing",
            (F.col("procedure_code") == "UNKNOWN").cast(IntegerType())
        )

    missing_diag_count = df_standardized.filter(F.col("is_diagnosis_missing") == 1).count()
    missing_proc_count = df_standardized.filter(F.col("is_procedure_missing") == 1).count()

    log_success(logger, f"Codes standardized — is_diagnosis_missing: {missing_diag_count}, "
                        f"is_procedure_missing: {missing_proc_count}")
    logger.info(f"After standardization: {df_standardized.count()} claims")

except Exception as e:
    log_error(logger, ErrorCodes.SLV_004, ErrorCodes.get_message(ErrorCodes.SLV_004), e)
    raise

### Join with providers

In [0]:
# Cell 8 — Step 5: Join with Providers (match rate: 100% from EDA)
log_step(logger, 6, "Join with Providers table (EDA: 100% match rate)")

try:
    logger.info("Performing left join with providers on provider_id")
    logger.info("EDA confirmed 100% match rate — no UNKNOWN specialty/location expected")

    df_with_providers = df_standardized.alias("c") \
        .join(
            df_providers.select(
                "provider_id", "specialty", "location"
                # doctor_name excluded — minimize PHI propagation in Silver
            ).alias("p"),
            on="provider_id",
            how="left"
        ) \
        .withColumn(
            "specialty",
            F.when(F.col("p.specialty").isNull(), F.lit("UNKNOWN"))
             .otherwise(F.col("p.specialty"))
        ) \
        .withColumn(
            "location",
            F.when(F.col("p.location").isNull(), F.lit("UNKNOWN"))
             .otherwise(F.col("p.location"))
        )

    after_count = df_with_providers.count()
    unknown_specialty = df_with_providers.filter(F.col("specialty") == "UNKNOWN").count()

    logger.info(f"After provider join: {after_count} rows")

    if after_count != df_standardized.count():
        log_warning(
            logger, ErrorCodes.SLV_005,
            f"Row count changed after provider join: "
            f"{df_standardized.count()} → {after_count}. "
            f"Check for duplicate provider_ids in providers table."
        )
    else:
        log_success(logger, f"Provider join complete — {after_count} rows, "
                            f"{unknown_specialty} UNKNOWN specialty values")

except Exception as e:
    log_error(logger, ErrorCodes.SLV_005, ErrorCodes.get_message(ErrorCodes.SLV_005), e)
    raise

### Join with diagnosis table

In [0]:
# Cell 9 — Step 6: Join with Diagnosis table (match rate: 83.33% — 1 unmatched from EDA)
log_step(logger, 7, "Join with Diagnosis table (EDA: 83.33% match, 1 unmatched code)")

try:
    logger.info("Performing left join with diagnosis on diagnosis_code")
    logger.info("EDA: 1 unmatched diagnosis_code — will show UNKNOWN category/severity")

    df_with_diagnosis = df_with_providers.alias("cp") \
        .join(
            df_diagnosis.select(
                "diagnosis_code", "category", "severity"
            ).alias("d"),
            on="diagnosis_code",
            how="left"
        ) \
        .withColumn(
            "diagnosis_category",
            F.when(F.col("d.category").isNull(), F.lit("UNKNOWN"))
             .otherwise(F.col("d.category"))
        ) \
        .withColumn(
            "diagnosis_severity",
            F.when(F.col("d.severity").isNull(), F.lit("UNKNOWN"))
             .otherwise(F.col("d.severity"))
        )

    after_count         = df_with_diagnosis.count()
    unknown_category    = df_with_diagnosis.filter(F.col("diagnosis_category") == "UNKNOWN").count()
    unknown_severity    = df_with_diagnosis.filter(F.col("diagnosis_severity") == "UNKNOWN").count()

    logger.info(f"After diagnosis join: {after_count} rows")
    logger.info(f"UNKNOWN diagnosis_category: {unknown_category} rows")
    logger.info(f"UNKNOWN diagnosis_severity: {unknown_severity} rows")

    if after_count != df_with_providers.count():
        log_warning(
            logger, ErrorCodes.SLV_006,
            f"Row count changed after diagnosis join: "
            f"{df_with_providers.count()} → {after_count}. "
            f"Check for duplicate diagnosis_codes in diagnosis table."
        )
    else:
        log_success(logger, f"Diagnosis join complete — {after_count} rows, "
                            f"{unknown_category} UNKNOWN categories")

except Exception as e:
    log_error(logger, ErrorCodes.SLV_006, ErrorCodes.get_message(ErrorCodes.SLV_006), e)
    raise

### Join with cost table

In [0]:
# Cell 10 — Step 7: Join with Cost table (match rate: 85.71% — 1 unmatched from EDA)
log_step(logger, 8, "Join with Cost table (EDA: 85.71% match, 1 unmatched code)")

try:
    logger.info("Performing left join with cost on procedure_code")
    logger.info("EDA: 1 unmatched procedure_code — will show 0.0 average_cost/expected_cost")

    pre_join_count = df_with_diagnosis.count()

    df_with_cost = df_with_diagnosis.alias("cpd") \
        .join(
            df_cost.select(
                "procedure_code", "average_cost", "expected_cost"
            ).alias("co"),
            on="procedure_code",
            how="left"
        ) \
        .withColumn(
            "average_cost",
            F.when(F.col("co.average_cost").isNull(), F.lit(0.0))
             .otherwise(F.col("co.average_cost"))
        ) \
        .withColumn(
            "expected_cost",
            F.when(F.col("co.expected_cost").isNull(), F.lit(0.0))
             .otherwise(F.col("co.expected_cost"))
        )

    after_count   = df_with_cost.count()
    zero_avg_cost = df_with_cost.filter(F.col("average_cost") == 0.0).count()
    zero_exp_cost = df_with_cost.filter(F.col("expected_cost") == 0.0).count()

    logger.info(f"Pre-join count  : {pre_join_count}")
    logger.info(f"Post-join count : {after_count}")
    logger.info(f"Zero average_cost (unmatched procedures) : {zero_avg_cost} rows")
    logger.info(f"Zero expected_cost (unmatched procedures): {zero_exp_cost} rows")

    if after_count != pre_join_count:
        log_warning(
            logger, ErrorCodes.SLV_007,
            f"Row count changed after cost join: "
            f"{pre_join_count} → {after_count}. "
            f"Check for duplicate procedure_codes in cost table."
        )
    else:
        log_success(
            logger,
            f"Cost join complete — {after_count} rows, "
            f"{zero_avg_cost} rows with zero average_cost"
        )

except Exception as e:
    log_error(logger, ErrorCodes.SLV_007, ErrorCodes.get_message(ErrorCodes.SLV_007), e)
    raise

### Financial denial rules

In [0]:
# Cell 10b — Tier 2: Financial denial rules — post cost join
# Rule: billed_amount > average_cost * 2 (extreme overbilling only)
# Adds 2 points to denial_points → may push borderline claims over threshold

log_step(logger, 9, "Tier 2 Financial Rules — Extreme Overbilling (>2x average_cost)")

try:
    logger.info("Applying R04: billed_amount > average_cost * 2 → extreme overbilling")
    logger.info("This adds 2 points to denial_points — may push borderline claims to denied")

    df_with_cost = df_with_cost \
        .withColumn(
            # Add 2 more points if extreme overbilling detected
            "denial_points",
            F.when(
                (F.col("average_cost") > 0) &
                (F.col("billed_amount") > F.col("average_cost") * 2),
                F.col("denial_points") + F.lit(2)
            ).otherwise(F.col("denial_points"))
        ) \
        .withColumn(
            # Update denial_reason_code only if not already set by Tier 1
            "denial_reason_code",
            F.when(
                F.col("denial_reason_code") != "NONE",
                F.col("denial_reason_code")
            ).when(
                (F.col("average_cost") > 0) &
                (F.col("billed_amount") > F.col("average_cost") * 2),
                F.lit("R04_EXTREME_OVERBILLING")
            ).otherwise(
                F.col("denial_reason_code")
            )
        ) \
        .withColumn(
            # Recalculate denial_flag based on updated points
            "denial_flag",
            F.when(
                F.col("denial_points") >= 3,
                F.lit(1)
            ).otherwise(F.lit(0)).cast(IntegerType())
        ) \
        .withColumn(
            # Recalculate denial_probability based on updated points
            "denial_probability",
            F.when(
                F.col("denial_points") == 0, F.lit(0.05)
            ).when(
                F.col("denial_points") == 1, F.lit(0.20)
            ).when(
                F.col("denial_points") == 2, F.lit(0.40)
            ).when(
                F.col("denial_points") == 3, F.lit(0.65)
            ).when(
                F.col("denial_points") == 4, F.lit(0.78)
            ).when(
                F.col("denial_points") == 5, F.lit(0.88)
            ).otherwise(
                F.lit(0.95)
            ).cast(DoubleType())
        ) \
        .withColumn(
            # Recalculate risk_tier after financial rules
            "risk_tier",
            F.when(
                F.col("denial_probability") >= 0.70, F.lit("HIGH")
            ).when(
                F.col("denial_probability") >= 0.40, F.lit("MEDIUM")
            ).otherwise(F.lit("LOW"))
        )

    # ── Overbilling stats ─────────────────────────────────────────────────────
    overbilled = df_with_cost.filter(
        (F.col("average_cost") > 0) &
        (F.col("billed_amount") > F.col("average_cost") * 2)
    ).count()

    # ── Final label distribution after ALL rules ──────────────────────────────
    total       = df_with_cost.count()
    denied      = df_with_cost.filter(F.col("denial_flag") == 1).count()
    approved    = total - denied
    rate        = round(denied / total * 100, 2)

    print("\n" + "=" * 60)
    print("FINAL LABEL DISTRIBUTION — ALL RULES APPLIED")
    print("=" * 60)
    print(f"  Denial threshold      : >= 3 points")
    print(f"  Extreme overbilling   : {overbilled} claims (billed > 2x avg_cost)")
    print(f"  Total claims          : {total}")
    print(f"  Denied  (flag=1)      : {denied}  ({rate}%)")
    print(f"  Approved (flag=0)     : {approved} ({round(100 - rate, 2)}%)")
    print("=" * 60)

    print("\n  Final denial reason breakdown:")
    df_with_cost.groupBy("denial_reason_code") \
        .count() \
        .orderBy(F.col("count").desc()) \
        .show(truncate=False)

    print("\n  Final risk tier distribution:")
    df_with_cost.groupBy("risk_tier") \
        .count() \
        .orderBy("risk_tier") \
        .show()

    print("\n  Points distribution (final):")
    df_with_cost.groupBy("denial_points") \
        .count() \
        .orderBy("denial_points") \
        .show()

    if rate < 20:
        log_warning(
            logger, ErrorCodes.EDA_005,
            f"Final denial rate ({rate}%) still below 20% after financial rules — "
            f"consider lowering DENIAL_THRESHOLD to 2 in Cell 6"
        )
    elif rate > 45:
        log_warning(
            logger, ErrorCodes.EDA_005,
            f"Final denial rate ({rate}%) above 45% — "
            f"consider raising DENIAL_THRESHOLD to 4 in Cell 6"
        )
    else:
        log_success(
            logger,
            f"Final label distribution healthy — "
            f"denial rate: {rate}% (target: ~30%)"
        )

except Exception as e:
    log_error(
        logger, ErrorCodes.SLV_003,
        "Tier 2 financial rule application failed", e
    )
    raise

### Silver metadata + claim_age_days + final column selection

In [0]:
# Cell 11 — Step 8: Silver metadata + claim_age_days + final column selection
log_step(logger, 10, "Silver Metadata + Final Column Selection")

try:
    # ── Compute claim_age_days ────────────────────────────────────────────────
    try:
        logger.info("Computing claim_age_days = current_date - claim_date")

        df_with_age = df_with_cost \
            .withColumn(
                "claim_age_days",
                F.datediff(F.current_date(), F.col("claim_date"))
            )

        null_age = df_with_age.filter(F.col("claim_age_days").isNull()).count()

        if null_age > 0:
            log_warning(
                logger, ErrorCodes.SLV_012,
                f"{null_age} rows have null claim_age_days — "
                f"likely due to unparseable claim_date values"
            )
        else:
            log_success(logger, f"claim_age_days computed for all {df_with_age.count()} rows")

    except Exception as age_err:
        log_error(
            logger, ErrorCodes.SLV_012,
            ErrorCodes.get_message(ErrorCodes.SLV_012), age_err
        )
        raise

    # ── Add Silver metadata + select final columns ────────────────────────────
    df_silver = df_with_age \
        .withColumn("_silver_timestamp", F.current_timestamp()) \
        .withColumn("_silver_layer", F.lit("silver")) \
        .select(
            # Core claim fields
            "claim_id", "patient_id", "provider_id",
            "claim_date", "claim_age_days",
            # Codes (standardized)
            "diagnosis_code", "procedure_code",
            # Amounts
            "billed_amount", "average_cost", "expected_cost",
            # Reference enrichment from joins
            "specialty", "location",
            "diagnosis_category", "diagnosis_severity",
            # ML feature flags
            "is_diagnosis_missing", "is_procedure_missing",
            # Labels — feeds Rule Engine + ML Model + RAG
            "denial_flag",          # binary target for ML training
            "denial_probability",   # soft score anchor for ML calibration
            "risk_tier",            # HIGH/MEDIUM/LOW for FastAPI response
            "denial_reason_code",   # RAG lookup key for policy retrieval
            # Metadata
            "_silver_timestamp", "_silver_layer"
        )

    final_count = df_silver.count()
    logger.info(f"Final Silver table: {final_count} rows × {len(df_silver.columns)} columns")

    # ── Schema validation ─────────────────────────────────────────────────────
    missing_cols = [c for c in REQUIRED_SILVER_COLS if c not in df_silver.columns]

    if missing_cols:
        log_error(
            logger, ErrorCodes.SLV_010,
            f"Schema validation failed — missing columns: {missing_cols}"
        )
        raise ValueError(
            ErrorCodes.format_error(
                ErrorCodes.SLV_010,
                f"Missing columns: {missing_cols}"
            )
        )
    else:
        log_success(logger, "Schema validation passed — all required columns present")

    print("\nFinal Silver Schema:")
    df_silver.printSchema()

except Exception as e:
    log_error(logger, ErrorCodes.SLV_010, ErrorCodes.get_message(ErrorCodes.SLV_010), e)
    raise

### Write Silver Delta table

In [0]:
# Cell 12 — Write Silver Delta table
log_step(logger, 11, "Writing Silver Delta table")

try:
    rows_to_write = df_silver.count()
    logger.info(f"Writing {rows_to_write} rows to: {SILVER_PATH}")
    logger.info("Partitioning by: denial_flag")

    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("denial_flag") \
        .save(SILVER_PATH)

    log_success(logger, f"Silver Delta table written — {rows_to_write} rows to {SILVER_PATH}")

except Exception as e:
    log_error(logger, ErrorCodes.SLV_008, ErrorCodes.get_message(ErrorCodes.SLV_008), e)
    raise

### Silver validation

In [0]:
# Cell 13 — Silver validation
log_step(logger, 12, "Silver Table Validation")

try:
    logger.info(f"Reading Silver table back from: {SILVER_PATH}")
    df_val    = spark.read.format("delta").load(SILVER_PATH)
    val_count = df_val.count()

    # ── Row count check ───────────────────────────────────────────────────────
    expected_count = df_silver.count()
    if val_count != expected_count:
        log_error(
            logger, ErrorCodes.SLV_009,
            f"Row count mismatch: wrote {expected_count}, read back {val_count}"
        )
        raise ValueError(ErrorCodes.format_error(ErrorCodes.SLV_009))
    else:
        log_success(logger, f"Row count validation passed: {val_count} rows")

    # ── Null checks on critical columns ───────────────────────────────────────
    critical_cols = [
        "claim_id", "patient_id", "provider_id",
        "billed_amount", "denial_flag",
        "denial_probability", "risk_tier", "denial_reason_code"
    ]

    print("\n" + "=" * 60)
    print("CRITICAL COLUMN NULL CHECK")
    print("=" * 60)

    for col in critical_cols:
        try:
            null_count = df_val.filter(F.col(col).isNull()).count()
            status     = "⚠️  WARNING" if null_count > 0 else "✅ OK"
            print(f"  {col:<25} nulls: {null_count:>5}  {status}")

            if null_count > 0:
                log_warning(
                    logger, ErrorCodes.SLV_010,
                    f"{null_count} null values found in '{col}' after Silver write"
                )
        except Exception as col_err:
            log_warning(
                logger, ErrorCodes.SLV_010,
                f"Could not validate column '{col}': {str(col_err)}"
            )

    # ── Full Silver validation report ─────────────────────────────────────────
    denied_count   = df_val.filter(F.col("denial_flag") == 1).count()
    approved_count = df_val.filter(F.col("denial_flag") == 0).count()
    denial_rate    = round(denied_count / val_count * 100, 2)
    avg_billed     = df_val.agg(F.avg("billed_amount")).collect()[0][0]

    print("\n" + "=" * 60)
    print("SILVER VALIDATION REPORT")
    print("=" * 60)
    print(f"  Total rows              : {val_count}")
    print(f"  Columns                 : {len(df_val.columns)}")
    print(f"  Denied claims           : {denied_count}  ({denial_rate}%)")
    print(f"  Approved claims         : {approved_count} ({round(100 - denial_rate, 2)}%)")
    print(f"  UNKNOWN diagnosis_code  : {df_val.filter(F.col('diagnosis_code') == 'UNKNOWN').count()}")
    print(f"  UNKNOWN procedure_code  : {df_val.filter(F.col('procedure_code') == 'UNKNOWN').count()}")
    print(f"  Avg billed amount       : ${round(avg_billed or 0, 2)}")
    print("=" * 60)

    print("\n  Denial reason breakdown:")
    df_val.groupBy("denial_reason_code") \
        .count() \
        .orderBy(F.col("count").desc()) \
        .show(truncate=False)

    print("\n  Risk tier distribution:")
    df_val.groupBy("risk_tier") \
        .count() \
        .orderBy("risk_tier") \
        .show()

    print("\n  Denial flag distribution:")
    df_val.groupBy("denial_flag") \
        .count() \
        .orderBy("denial_flag") \
        .show()

    log_success(logger, "Silver validation complete — all checks passed")

except Exception as e:
    log_error(logger, ErrorCodes.SLV_009, ErrorCodes.get_message(ErrorCodes.SLV_009), e)
    raise

finally:
    log_end(logger, "silver_cleaning")

In [0]:
from pyspark.sql import functions as F

BASE_PATH   = "/Volumes/workspace/default/myvol"
SILVER_PATH = f"{BASE_PATH}/silver/claims_cleaned/"

df_silver = spark.read.format("delta").load(SILVER_PATH)

print(f"Total rows : {df_silver.count()}")
print(f"Columns    : {len(df_silver.columns)}")
print(f"\nColumns list: {df_silver.columns}")

df_silver.display(20, truncate=False)